In [10]:
!pip3 install happybase

2026-07-26 00:47:47 | INFO | Closing down clientserver connection



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [11]:
# =============================================================================
# Phase 8 — HBase Storage Layer
# Processed Streaming Data (HDFS Parquet) → HBase
# Craigslist Used Vehicle Dataset
# =============================================================================

import logging
import time
import hashlib
import subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("VehicleHBase")

In [12]:
# =============================================================================
# CONFIG
# =============================================================================

STREAMING_OUTPUT = "hdfs://127.0.0.1:9000/user/vehicle_market/streaming/processed"

HBASE_NAMESPACE   = "vehicle_market"
HBASE_TABLE       = "vehicle_listings"
HBASE_FULL_TABLE  = f"{HBASE_NAMESPACE}:{HBASE_TABLE}"

# Column families — each groups related attributes together
# Matches the logical groupings used across all dashboards
CF_VEHICLE   = "vehicle"      # core vehicle attributes
CF_PRICING   = "pricing"      # price, category, band, per-mile
CF_LOCATION  = "location"     # state, lat, long, region
CF_FEATURES  = "features"     # condition, drive, fuel, transmission
CF_METADATA  = "metadata"     # processing timestamps, flags, stream info

HBASE_WRITE_BATCH_SIZE = 1000   # rows per HBase batch write

# HBase shell executable — make configurable for portability
HBASE_CMD = "hbase"

In [13]:
# =============================================================================
# 1. SPARK SESSION
# HBase connectivity via SHC (Spark-HBase Connector)
# or hbase-spark connector depending on your local installation
# =============================================================================

spark = SparkSession.builder \
    .appName("VehicleMarket_HBase") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

pipeline_start = time.time()
log.info("="*65)
log.info("  Phase 8 — HBase Storage Layer")
log.info("  HDFS Parquet → HBase NoSQL")
log.info("="*65)

2026-07-26 00:47:48 | INFO | =================================================================
2026-07-26 00:47:48 | INFO |   Phase 8 — HBase Storage Layer
2026-07-26 00:47:48 | INFO |   HDFS Parquet → HBase NoSQL
2026-07-26 00:47:48 | INFO | =================================================================


In [14]:
# =============================================================================
# 2. VALIDATE HBASE SERVICE
# =============================================================================

log.info("Validating HBase service...")

def run_hbase_shell(commands, timeout=30):
    """
    Execute HBase shell commands via subprocess.
    commands: list of HBase shell statements as strings
    Returns (success, stdout, stderr)
    """
    shell_input = "\n".join(commands) + "\nexit\n"
    try:
        result = subprocess.run(
            [HBASE_CMD, "shell"],
            input=shell_input,
            capture_output=True,
            text=True,
            timeout=timeout
        )
        return result.returncode == 0, result.stdout, result.stderr
    except FileNotFoundError:
        return False, "", (
            f"'{HBASE_CMD}' not found. "
            f"Set HBASE_CMD to the full path of the HBase executable."
        )
    except subprocess.TimeoutExpired:
        return False, "", "HBase shell command timed out."
    except Exception as e:
        return False, "", str(e)

# Verify HBase is reachable with a simple status command
log.info("Validating HBase service...")

try:
    import happybase

    test_conn = happybase.Connection("localhost", port=9090)
    test_conn.open()

    log.info("  ✓ HBase Thrift server reachable.")
    log.info(f"  Tables found: {len(test_conn.tables())}")

    test_conn.close()

except Exception as e:
    raise ConnectionError(
        "Unable to connect to HBase via HappyBase.\n"
        "Ensure the following services are running:\n"
        "  • HMaster\n"
        "  • HRegionServer\n"
        "  • HBase Thrift Server\n"
        f"\nDetails: {e}"
    )



2026-07-26 00:47:48 | INFO | Validating HBase service...
2026-07-26 00:47:48 | INFO | Validating HBase service...
2026-07-26 00:47:48 | INFO |   ✓ HBase Thrift server reachable.
2026-07-26 00:47:48 | INFO | Error while sending or receiving.
Traceback (most recent call last):
  File "/Users/huntstar/spark-3.5.0/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
BrokenPipeError: [Errno 32] Broken pipe
2026-07-26 00:47:48 | INFO | Closing down clientserver connection
2026-07-26 00:47:48 | INFO | Exception while sending command.
Traceback (most recent call last):
  File "/Users/huntstar/spark-3.5.0/python/lib/py4j-0.10.9.7-src.zip/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
BrokenPipeError: [Errno 32] Broken pipe

During handling of the above exception, another 

In [15]:
# =============================================================================
# 3. VALIDATE HDFS STREAMING OUTPUT EXISTS
# =============================================================================

log.info("Validating HDFS streaming output...")

sc  = spark.sparkContext
fs  = sc._jvm.org.apache.hadoop.fs.FileSystem.get(
    sc._jsc.hadoopConfiguration()
)
hdfs_path = sc._jvm.org.apache.hadoop.fs.Path(STREAMING_OUTPUT)

if not fs.exists(hdfs_path):
    raise FileNotFoundError(
        f"Streaming output not found at: {STREAMING_OUTPUT}\n"
        f"Run Phase 7 (Spark Structured Streaming) before this notebook."
    )

log.info(f"  Streaming output confirmed: {STREAMING_OUTPUT}")


2026-07-26 00:47:48 | INFO | Validating HDFS streaming output...
2026-07-26 00:47:48 | INFO |   Streaming output confirmed: hdfs://127.0.0.1:9000/user/vehicle_market/streaming/processed


In [16]:
# =============================================================================
# 4. CREATE HBASE NAMESPACE AND TABLE
# =============================================================================

log.info(f"Setting up HBase namespace and table...")

# Create namespace
ns_ok, ns_out, ns_err = run_hbase_shell([
    f"create_namespace '{HBASE_NAMESPACE}'"
])
# Namespace may already exist — that's fine
if "already exists" in ns_out or ns_ok:
    log.info(f"  Namespace '{HBASE_NAMESPACE}' ready.")
else:
    log.warning(f"  Namespace creation note: {ns_err.strip()}")

# Disable and drop table if it exists (clean slate on re-run)
run_hbase_shell([
    f"disable '{HBASE_FULL_TABLE}' if exists('{HBASE_FULL_TABLE}')",
    f"drop '{HBASE_FULL_TABLE}' if exists('{HBASE_FULL_TABLE}')"
])

# Create table with column families
# TTL and COMPRESSION can be added for production use
create_cmd = (
    f"create '{HBASE_FULL_TABLE}', "
    f"{{NAME => '{CF_VEHICLE}',  VERSIONS => 1}}, "
    f"{{NAME => '{CF_PRICING}',  VERSIONS => 1}}, "
    f"{{NAME => '{CF_LOCATION}', VERSIONS => 1}}, "
    f"{{NAME => '{CF_FEATURES}', VERSIONS => 1}}, "
    f"{{NAME => '{CF_METADATA}', VERSIONS => 1}}"
)

create_ok, create_out, create_err = run_hbase_shell([create_cmd])

if not create_ok and "already exists" not in create_err:
    raise RuntimeError(
        f"Failed to create HBase table '{HBASE_FULL_TABLE}'.\n"
        f"Details: {create_err}"
    )

log.info(f"  Table '{HBASE_FULL_TABLE}' ready.")
log.info(f"  Column families: {CF_VEHICLE}, {CF_PRICING}, "
         f"{CF_LOCATION}, {CF_FEATURES}, {CF_METADATA}")


2026-07-26 00:47:48 | INFO | Setting up HBase namespace and table...
2026-07-26 00:47:51 | INFO |   Namespace 'vehicle_market' ready.
2026-07-26 00:47:55 | INFO |   Table 'vehicle_market:vehicle_listings' ready.
2026-07-26 00:47:55 | INFO |   Column families: vehicle, pricing, location, features, metadata


In [17]:
# =============================================================================
# 5. LOAD PROCESSED STREAMING DATA FROM HDFS
# =============================================================================

log.info("Loading processed streaming data from HDFS...")

df = spark.read.parquet(STREAMING_OUTPUT)

# Cache avoids re-reading the Parquet dataset across multiple actions:
# schema validation, row key generation UDF application, and toLocalIterator()
df.cache()
record_count_in = df.count()    # triggers cache materialization

log.info(f"  Records loaded  : {record_count_in}")
log.info(f"  Columns         : {len(df.columns)}")



2026-07-26 00:47:55 | INFO | Loading processed streaming data from HDFS...
2026-07-26 00:47:56 | INFO |   Records loaded  : 38712                          
2026-07-26 00:47:56 | INFO |   Columns         : 42


In [18]:
# =============================================================================
# 6. SCHEMA VALIDATION
# =============================================================================

log.info("Validating schema...")

REQUIRED_COLS = [
    "id", "manufacturer", "model", "year", "price", "odometer",
    "fuel", "transmission", "condition", "title_status", "drive",
    "state", "lat", "long", "vehicle_age", "age_group",
    "price_category", "mileage_category", "price_per_mile",
    "posting_year", "posting_month", "posting_season",
    "is_luxury", "is_alternative_fuel", "vehicle_segment",
    "clean_title", "is_salvage", "is_automatic", "price_band",
    "posting_period", "is_weekend", "depreciation_index",
    "processed_at"
]

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(
        f"Processed dataset is missing columns: {missing_cols}. "
        f"Re-run Phase 7 before proceeding."
    )

log.info(f"  All {len(REQUIRED_COLS)} required columns present.")

2026-07-26 00:47:56 | INFO | Validating schema...
2026-07-26 00:47:56 | INFO |   All 33 required columns present.


In [19]:
# =============================================================================
# 7. ROW KEY DESIGN
#
# Row key format: {state}#{posting_year}#{padded_price}#{id_hash}
#
# Design rationale:
# — state prefix enables efficient state-level range scans
# — posting_year groups records chronologically within each state
# — padded_price (zero-padded to 7 digits) enables price-range scans
#   within state+year without full table scans
# — id_hash suffix (first 8 chars of MD5) ensures uniqueness
#   and distributes records within each partition evenly
#
# This avoids hotspotting by distributing writes across regions
# rather than appending sequentially to a single region server.
# =============================================================================

log.info("Designing row keys...")

def make_row_key(state, posting_year, price, record_id):
    """
    Construct deterministic HBase row key.
    Zero-pads price to 7 digits for lexicographic range scan support.
    Appends 8-char MD5 hash of record_id for uniqueness guarantee.

    Note: MD5 is used here solely to generate a short, deterministic
    suffix for row key uniqueness. It is not used for cryptographic
    security purposes.
    """
    id_hash   = hashlib.md5(str(record_id).encode()).hexdigest()[:8]
    price_pad = str(int(price)).zfill(7) if price else "0000000"
    return f"{state}#{posting_year}#{price_pad}#{id_hash}"

# Register as Spark UDF for DataFrame-level row key generation
make_row_key_udf = F.udf(make_row_key)

df = df.withColumn(
    "row_key",
    make_row_key_udf(
        F.col("state").cast("string"),
        F.col("posting_year").cast("string"),
        F.col("price").cast("string"),
        F.col("id").cast("string")
    )
)

log.info("  Row key format: {state}#{posting_year}#{price_padded7}#{id_hash8}")
log.info("  Sample row keys:")
df.select("row_key", "state", "posting_year", "price", "id") \
  .show(5, truncate=False)


2026-07-26 00:47:56 | INFO | Designing row keys...
2026-07-26 00:47:56 | INFO |   Row key format: {state}#{posting_year}#{price_padded7}#{id_hash8}
2026-07-26 00:47:56 | INFO |   Sample row keys:


+------------------------+-----+------------+-----+----------+
|row_key                 |state|posting_year|price|id        |
+------------------------+-----+------------+-----+----------+
|ca#2021#0001800#ca721fb4|ca   |2021        |1800 |7311213622|
|ca#2021#0008500#9f451313|ca   |2021        |8500 |7313036721|
|ca#2021#0016990#d9251047|ca   |2021        |16990|7315836862|
|ca#2021#0025999#62987664|ca   |2021        |25999|7315990880|
|ca#2021#0017950#fdad2dad|ca   |2021        |17950|7316367549|
+------------------------+-----+------------+-----+----------+
only showing top 5 rows



In [20]:
# =============================================================================
# 8. COLUMN FAMILY MAPPING
# Maps each DataFrame column to its HBase column family
# =============================================================================

COLUMN_FAMILY_MAP = {
    # vehicle — core vehicle identity attributes
    "id"                 : CF_VEHICLE,
    "manufacturer"       : CF_VEHICLE,
    "model"              : CF_VEHICLE,
    "year"               : CF_VEHICLE,
    "type"               : CF_VEHICLE,
    "vehicle_age"        : CF_VEHICLE,
    "age_group"          : CF_VEHICLE,
    "vehicle_segment"    : CF_VEHICLE,

    # pricing — all price-related fields
    "price"              : CF_PRICING,
    "price_category"     : CF_PRICING,
    "price_band"         : CF_PRICING,
    "price_per_mile"     : CF_PRICING,
    "price_per_mile_live": CF_PRICING,
    "is_high_value"      : CF_PRICING,

    # location — geographic fields
    "state"              : CF_LOCATION,
    "lat"                : CF_LOCATION,
    "long"               : CF_LOCATION,

    # features — vehicle condition and specification
    "odometer"           : CF_FEATURES,
    "fuel"               : CF_FEATURES,
    "transmission"       : CF_FEATURES,
    "condition"          : CF_FEATURES,
    "title_status"       : CF_FEATURES,
    "drive"              : CF_FEATURES,
    "paint_color"        : CF_FEATURES,
    "mileage_category"   : CF_FEATURES,
    "is_luxury"          : CF_FEATURES,
    "is_alternative_fuel": CF_FEATURES,
    "is_automatic"       : CF_FEATURES,
    "is_salvage"         : CF_FEATURES,
    "clean_title"        : CF_FEATURES,

    # metadata — processing and streaming audit fields
    "posting_date"       : CF_METADATA,
    "posting_year"       : CF_METADATA,
    "posting_month"      : CF_METADATA,
    "posting_season"     : CF_METADATA,
    "posting_period"     : CF_METADATA,
    "is_weekend"         : CF_METADATA,
    "processed_at"       : CF_METADATA,
    "kafka_timestamp"    : CF_METADATA,
    "kafka_partition"    : CF_METADATA,
    "kafka_offset"       : CF_METADATA,

    # pricing — engineered analytical feature
    "depreciation_index" : CF_PRICING,
}

In [21]:
# =============================================================================
# 9. WRITE TO HBASE VIA HAPPYBASE (Python HBase client)
#
# happybase connects to HBase Thrift server.
# Ensure Thrift server is running:
#   $HBASE_HOME/bin/hbase thrift start &
#
# Alternative: use hbase-python or phonenix depending on your setup.
# =============================================================================

try:
    import happybase
except ImportError:
    raise ImportError(
        "happybase is not installed.\n"
        "Install it with: pip install happybase\n"
        "Also ensure HBase Thrift server is running:\n"
        "  $HBASE_HOME/bin/hbase thrift start &"
    )

log.info("Connecting to HBase via Thrift...")

try:
    hbase_connection = happybase.Connection("localhost")
    hbase_connection.open()
    log.info("  HBase Thrift connection established.")
except Exception as e:
    raise ConnectionError(
        f"Cannot connect to HBase Thrift server.\n"
        f"Start it with: $HBASE_HOME/bin/hbase thrift start &\n"
        f"Details: {e}"
    )




2026-07-26 00:47:57 | INFO | Connecting to HBase via Thrift...
2026-07-26 00:47:57 | INFO |   HBase Thrift connection established.


In [22]:
# =============================================================================
# 10. BATCH WRITE TO HBASE
# =============================================================================

log.info(f"Writing records to HBase table: {HBASE_FULL_TABLE}...")
log.info(f"  Batch size: {HBASE_WRITE_BATCH_SIZE}")

write_start      = time.time()
records_written  = 0
records_failed   = 0
batch_number     = 0

# Collect to driver for HBase Put operations
# HBase writes are inherently single-node (Thrift client is not distributed)
# toLocalIterator() streams partitions one at a time to avoid OOM
rows_iter = df.toLocalIterator()

table  = hbase_connection.table(HBASE_FULL_TABLE.encode())
batch  = table.batch(batch_size=HBASE_WRITE_BATCH_SIZE)

try:
    for row in rows_iter:
        row_dict = row.asDict()
        row_key  = row_dict.get("row_key", "")

        if not row_key:
            records_failed += 1
            continue

        # Build HBase Put data — {b"cf:col": b"value"}
        put_data = {}
        for col, cf in COLUMN_FAMILY_MAP.items():
            val = row_dict.get(col)
            if val is not None:
                col_key           = f"{cf}:{col}".encode()
                put_data[col_key] = str(val).encode()

        if not put_data:
            records_failed += 1
            continue

        try:
            batch.put(row_key.encode(), put_data)
            records_written += 1

            # Log progress every HBASE_WRITE_BATCH_SIZE records
            if records_written % HBASE_WRITE_BATCH_SIZE == 0:
                batch_number += 1
                elapsed    = round(time.time() - write_start, 1)
                throughput = round(
                    records_written / elapsed, 1
                ) if elapsed > 0 else 0
                log.info(
                    f"  Batch {batch_number:>4} | "
                    f"Written: {records_written:>6} | "
                    f"Failed: {records_failed:>4} | "
                    f"Elapsed: {elapsed:>6}s | "
                    f"Throughput: {throughput} rec/s"
                )

        except Exception as e:
            records_failed += 1
            log.error(f"  ✗ Put failed for row_key '{row_key}': {e}")

    # Send any remaining records in the final partial batch
    try:
        batch.send()
        log.info("  Final batch flushed successfully.")
    except Exception as e:
        log.error(f"  ✗ Final batch flush failed: {e}")
        # Count unflushed records as failed
        # Exact count unavailable from happybase batch internals
        # so we log a warning for the validation report
        log.warning(
            "  ⚠ Some records in the final batch may not have been "
            "written. Check HBase logs for details."
        )

except Exception as e:
    log.error(f"  ✗ Batch write error: {e}")

finally:
    hbase_connection.close()
    log.info("  HBase connection closed.")

write_end  = time.time()
write_time = round(write_end - write_start, 2)
avg_throughput = round(
    records_written / write_time, 1
) if write_time > 0 else 0

log.info(f"  Write complete — {records_written} written, "
         f"{records_failed} failed in {write_time}s")

2026-07-26 00:47:57 | INFO | Writing records to HBase table: vehicle_market:vehicle_listings...
2026-07-26 00:47:57 | INFO |   Batch size: 1000
2026-07-26 00:47:58 | INFO |   Batch    1 | Written:   1000 | Failed:    0 | Elapsed:    0.7s | Throughput: 1428.6 rec/s
2026-07-26 00:47:58 | INFO |   Batch    2 | Written:   2000 | Failed:    0 | Elapsed:    0.9s | Throughput: 2222.2 rec/s
2026-07-26 00:47:58 | INFO |   Batch    3 | Written:   3000 | Failed:    0 | Elapsed:    1.0s | Throughput: 3000.0 rec/s
2026-07-26 00:47:59 | INFO |   Batch    4 | Written:   4000 | Failed:    0 | Elapsed:    1.1s | Throughput: 3636.4 rec/s
2026-07-26 00:47:59 | INFO |   Batch    5 | Written:   5000 | Failed:    0 | Elapsed:    1.2s | Throughput: 4166.7 rec/s
2026-07-26 00:47:59 | INFO |   Batch    6 | Written:   6000 | Failed:    0 | Elapsed:    1.3s | Throughput: 4615.4 rec/s
2026-07-26 00:47:59 | INFO |   Batch    7 | Written:   7000 | Failed:    0 | Elapsed:    1.5s | Throughput: 4666.7 rec/s
2026-07-2

In [23]:
# =============================================================================
# 11. POST-WRITE VALIDATION
# Reconnect and scan table to verify records landed correctly
# =============================================================================

log.info("Running post-write validation...")

try:
    val_connection = happybase.Connection("localhost")
    val_connection.open()
    val_table = val_connection.table(HBASE_FULL_TABLE.encode())

    # --- Row count via scan ---
    scanned_count = sum(1 for _ in val_table.scan())
    log.info(f"  Scanned row count : {scanned_count}")

    if scanned_count < records_written:
        log.warning(
            f"  ⚠ Scanned rows ({scanned_count}) < "
            f"records written ({records_written}). "
            f"Possible partial write — check HBase logs."
        )
    else:
        log.info("  Row count validation passed.")

    # --- Sample record retrieval ---
    log.info("  Sample records (first 3):")
    for idx, (key, data) in enumerate(val_table.scan(limit=3)):
        log.info(f"    Row {idx+1}: key={key.decode()}")
        for col, val in list(data.items())[:5]:   # show first 5 columns only
            log.info(f"      {col.decode()}: {val.decode()}")

    # --- Column family presence check ---
    log.info("  Checking column families in sample record...")
    first_row = next(val_table.scan(limit=1), None)
    if first_row:
        key, data       = first_row
        col_families    = set(
            c.decode().split(":")[0] for c in data.keys()
        )
        expected_families = {
            CF_VEHICLE, CF_PRICING, CF_LOCATION, CF_FEATURES, CF_METADATA
        }
        missing_families  = expected_families - col_families
        if missing_families:
            log.warning(f"  ⚠ Missing column families: {missing_families}")
        else:
            log.info(
                f"  All column families present: {sorted(col_families)}"
            )

    # --- Lookup query — retrieve records for a specific state ---
    log.info("  Lookup query: scanning records for state 'ca'...")
    ca_rows = list(val_table.scan(
        row_prefix=b"ca#",
        limit=5,
        columns=[
            f"{CF_VEHICLE}:manufacturer".encode(),
            f"{CF_PRICING}:price".encode(),
            f"{CF_LOCATION}:state".encode()
        ]
    ))
    log.info(f"  Found {len(ca_rows)} California records (limit 5):")
    for key, data in ca_rows:
        mfr   = data.get(f"{CF_VEHICLE}:manufacturer".encode(), b"N/A").decode()
        price = data.get(f"{CF_PRICING}:price".encode(), b"N/A").decode()
        log.info(f"    key={key.decode()} | manufacturer={mfr} | price=${price}")

    # --- Lookup query — price range scan within California 2021 ---
    log.info("  Range scan: California 2021 vehicles priced $10k–$20k...")
    range_rows = list(val_table.scan(
        row_start=b"ca#2021#0010000",
        row_stop=b"ca#2021#0020000",
        limit=5,
        columns=[
            f"{CF_VEHICLE}:manufacturer".encode(),
            f"{CF_PRICING}:price".encode(),
            f"{CF_VEHICLE}:model".encode()
        ]
    ))
    log.info(f"  Found {len(range_rows)} records in range:")
    for key, data in range_rows:
        mfr   = data.get(f"{CF_VEHICLE}:manufacturer".encode(), b"N/A").decode()
        model = data.get(f"{CF_VEHICLE}:model".encode(), b"N/A").decode()
        price = data.get(f"{CF_PRICING}:price".encode(), b"N/A").decode()
        log.info(f"    {mfr} {model} — ${price}")

    val_connection.close()
    validation_status = "✓ PASSED"

except Exception as e:
    scanned_count     = 0
    validation_status = f"✗ FAILED: {e}"
    log.error(f"  Validation failed: {e}")


2026-07-26 00:48:03 | INFO | Running post-write validation...
2026-07-26 00:48:04 | INFO |   Scanned row count : 9679
2026-07-26 00:48:04 | WARNING |   ⚠ Scanned rows (9679) < records written (38712). Possible partial write — check HBase logs.
2026-07-26 00:48:04 | INFO |   Sample records (first 3):
2026-07-26 00:48:04 | INFO |     Row 1: key=ak#2021#0001850#9d03e75e
2026-07-26 00:48:04 | INFO |       features:clean_title: True
2026-07-26 00:48:04 | INFO |       features:condition: unknown
2026-07-26 00:48:04 | INFO |       features:drive: unknown
2026-07-26 00:48:04 | INFO |       features:fuel: gas
2026-07-26 00:48:04 | INFO |       features:is_alternative_fuel: False
2026-07-26 00:48:04 | INFO |     Row 2: key=al#2021#0000199#3d0ed108
2026-07-26 00:48:04 | INFO |       features:clean_title: True
2026-07-26 00:48:04 | INFO |       features:condition: unknown
2026-07-26 00:48:04 | INFO |       features:drive: unknown
2026-07-26 00:48:04 | INFO |       features:fuel: gas
2026-07-26 00:

In [24]:
# =============================================================================
# 12. HBASE TABLE SUMMARY VIA SHELL
# =============================================================================

log.info("Retrieving HBase table summary...")

desc_ok, desc_out, _ = run_hbase_shell([
    f"describe '{HBASE_FULL_TABLE}'"
])
if desc_ok:
    for line in desc_out.strip().split("\n"):
        if any(x in line for x in ["NAME", "VERSIONS", "Table", "column"]):
            log.info(f"  {line.strip()}")

count_ok, count_out, _ = run_hbase_shell([
    f"count '{HBASE_FULL_TABLE}', INTERVAL => 100000"
])
if count_ok:
    for line in count_out.strip().split("\n"):
        if "row(s)" in line or "Current count" in line:
            log.info(f"  {line.strip()}")

2026-07-26 00:48:04 | INFO | Retrieving HBase table summary...
2026-07-26 00:48:06 | INFO |   Table vehicle_market:vehicle_listings is ENABLED
2026-07-26 00:48:06 | INFO |   {NAME => 'features', BLOOMFILTER => 'ROW', VERSIONS => '1', IN_MEMORY => 'false'
2026-07-26 00:48:06 | INFO |   , COMPRESSION => 'NONE', MIN_VERSIONS => '0', BLOCKCACHE => 'true', BLOCKSIZE =>
2026-07-26 00:48:06 | INFO |   {NAME => 'location', BLOOMFILTER => 'ROW', VERSIONS => '1', IN_MEMORY => 'false'
2026-07-26 00:48:06 | INFO |   , COMPRESSION => 'NONE', MIN_VERSIONS => '0', BLOCKCACHE => 'true', BLOCKSIZE =>
2026-07-26 00:48:06 | INFO |   {NAME => 'metadata', BLOOMFILTER => 'ROW', VERSIONS => '1', IN_MEMORY => 'false'
2026-07-26 00:48:06 | INFO |   , COMPRESSION => 'NONE', MIN_VERSIONS => '0', BLOCKCACHE => 'true', BLOCKSIZE =>
2026-07-26 00:48:06 | INFO |   {NAME => 'pricing', BLOOMFILTER => 'ROW', VERSIONS => '1', IN_MEMORY => 'false',
2026-07-26 00:48:06 | INFO |   COMPRESSION => 'NONE', MIN_VERSIONS => '0'

In [25]:
# =============================================================================
# 13. UNPERSIST
# =============================================================================

df.unpersist()
log.info("HDFS DataFrame cache released.")

2026-07-26 00:48:08 | INFO | HDFS DataFrame cache released.


In [26]:
# =============================================================================
# 14. HBASE VALIDATION REPORT
# =============================================================================

pipeline_end   = time.time()
execution_time = round(pipeline_end - pipeline_start, 2)

write_status = "✓ SUCCESS" if records_failed == 0 \
    else f"⚠ PARTIAL ({records_failed} records failed)"

log.info("\n" + "="*65)
log.info("       HBASE STORAGE — VALIDATION REPORT")
log.info("="*65)
log.info(f"  Execution Time          : {execution_time}s")
log.info(f"  Write Time              : {write_time}s")
log.info(f"  HBase Table             : {HBASE_FULL_TABLE}")
log.info(f"  Column Families         : {CF_VEHICLE}, {CF_PRICING}, "
         f"{CF_LOCATION}, {CF_FEATURES}, {CF_METADATA}")
log.info(f"  Row Key Format          : state#year#price(7d)#id_hash(8c)")
log.info(f"  Source                  : {STREAMING_OUTPUT}")
log.info(f"  Records Read (HDFS)     : {record_count_in}")
log.info(f"  Records Written (HBase) : {records_written}")
log.info(f"  Records Failed          : {records_failed}")
log.info(f"  Records Scanned (Verify): {scanned_count}")
log.info(f"  Batch Size              : {HBASE_WRITE_BATCH_SIZE}")
log.info(f"  Avg Throughput          : {avg_throughput} records/sec")
log.info(f"  Write Status            : {write_status}")
log.info(f"  Validation Status       : {validation_status}")
log.info("")

if records_failed == 0 and validation_status.startswith("✓"):
    log.info("  ✓ HBase Storage complete.")
    log.info("  ✓ All records written and verified.")
    log.info("  ✓ Pipeline complete — ready for Tableau dashboards.")
else:
    log.info("  ⚠ Storage completed with issues. Review errors above.")

log.info("="*65)

2026-07-26 00:48:08 | INFO | 
2026-07-26 00:48:08 | INFO |        HBASE STORAGE — VALIDATION REPORT
2026-07-26 00:48:08 | INFO | =================================================================
2026-07-26 00:48:08 | INFO |   Execution Time          : 20.58s
2026-07-26 00:48:08 | INFO |   Write Time              : 5.27s
2026-07-26 00:48:08 | INFO |   HBase Table             : vehicle_market:vehicle_listings
2026-07-26 00:48:08 | INFO |   Column Families         : vehicle, pricing, location, features, metadata
2026-07-26 00:48:08 | INFO |   Row Key Format          : state#year#price(7d)#id_hash(8c)
2026-07-26 00:48:08 | INFO |   Source                  : hdfs://127.0.0.1:9000/user/vehicle_market/streaming/processed
2026-07-26 00:48:08 | INFO |   Records Read (HDFS)     : 38712
2026-07-26 00:48:08 | INFO |   Records Written (HBase) : 38712
2026-07-26 00:48:08 | INFO |   Records Failed          : 0
2026-07-26 00:48:08 | INFO |   Records Scanned (Verify): 9679
2026-07-26 00:48:08 | INFO | 